In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from regpy.operators import CoordinateProjection 
from regpy.operators.convolution import ConvolutionOperator, GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import RegularizationSetting
from regpy.solvers.linear.tikhonov import TikhonovCG
from regpy.solvers.linear.landweber import Landweber
from regpy.hilbert import L2, HmDomain
import regpy.stoprules as rules
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

In [ ]:
def comparison_plot(grid,truth,reco,title_right='exact',title_left='reconstruction'):
    plt.rcParams.update({'font.size': 22})
    extent = [grid.axes[0][0],grid.axes[0][-1], grid.axes[1][0], grid.axes[1][-1]]
    maxval = np.max(truth[:]); minval = np.min(truth[:])
    mycmap = plt.cm.get_cmap('hot')
    mycmap.set_over((0,0,1.,1.))  # use blue as access color for large values
    mycmap.set_under((0,1,0,1.))  # use green as access color for small values
    fig, (ax1,ax2) = plt.subplots(1,2,figsize = (22,8))
    im1= ax1.imshow(reco.T,extent=extent,origin='lower',
                    vmin=1.1*minval-0.1*maxval, vmax =1.1*maxval-0.1*minval,
                    cmap=mycmap
                    )
    ax1.title.set_text(title_left)
    fig.colorbar(im1,extend='both')
    im2= ax2.imshow(truth.T,extent=extent, origin='lower',
                    vmin=1.1*minval-0.1*maxval, vmax =1.1*maxval-0.1*minval,
                    cmap=mycmap
                    )
    ax2.title.set_text(title_right)
    fig.colorbar(im2,extend='both',orientation='vertical')
    


In [ ]:
grid = UniformGridFcts((-1, 1, 256), (-1.5, 1, 256),dtype = float, periodic = True)
X = grid.coords[0]; Y = grid.coords[1]
cross = 1.0*np.logical_or((abs(X)<0.01) * (abs(Y)<0.3),(abs(X)<0.3) * (abs(Y)<0.01)) 
rad = np.sqrt(X**2 + Y**2)
ring = 1.0*np.logical_and(rad>=0.9, rad<=0.95)
smallbox = (abs(X+0.55)<=0.05) * (abs(Y-0.55)<=0.05)
bubbles = (1.+np.sin(50/(X+1.3)))*np.exp(-((Y+1.25)/0.1)**2)*(X>-0.8)*(X<0.8)

hills = 200*(1+  np.sin(100*(Y*X+X**2)))*np.exp(-(X/0.3)**2 - ((Y+0.25)/0.4)**2)
#np.sin(12*X-10*Y+20*Y**2)*(rad<=0.95) #*np.exp(-(X/0.3)**2 - (Y/0.4)**2)

objects = 200*(ring + 2.0*cross + 1.5*smallbox + bubbles)
exact_sol = hills #objects
 
comparison_plot(grid,hills,objects)
#extent = [grid.axes[0][0],grid.axes[0][-1], grid.axes[1][0], grid.axes[1][-1]]
#plt.imshow(exact_sol,extent= extent,origin='lower',cmap = 'hot');plt.colorbar()


In [ ]:
a=0.05
conv = GaussianBlur(grid,a)

blur = conv(exact_sol)
data = np.random.poisson(blur)
#noise = grid.randn()
#noise_level = 0.2
#data = blur + (noise_level *np.linalg.norm(blur[:])/np.linalg.norm(noise[:]))* noise
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

In [ ]:
alpha = 3e-4
otf = conv.fourier_multiplier
reco_op= ConvolutionOperator(grid, np.conj(otf)/(np.abs(otf)**2+alpha))
reco = reco_op(data)
print('relative reconstruction error:', np.linalg.norm(reco[:]-exact_sol[:])/np.linalg.norm(exact_sol[:]))
comparison_plot(grid,exact_sol,reco,title_left='Tikhonov Fourier space')

In [ ]:
from regpy.solvers import Solver
from regpy.stoprules import CountIterations
class TikhonovAlphaGrid(Solver):
    def __init__(self,setting, data, alphas, **kwargs):
        self._alphas = alphas
        self.setting = setting
        self.data = data
        self.max_inner_iter = 1000
        self.kwargs = kwargs
        print('kwargs',kwargs)
        self.x = setting.op.domain.zeros()
        self.y = setting.op.codomain.zeros()
        super().__init__()

    def _next(self):
        print('start next')
        alpha = next(self._alphas)
        inner_stoprule = rules.CountIterations(max_iterations=self.max_inner_iter)
        tikhcg =TikhonovCG(self.setting,self.data,alpha)
        self.x, self.y = tikhcg.run(inner_stoprule)

In [ ]:
setting = RegularizationSetting(op=conv, penalty=L2, data_fid=L2)
#solver = TikhonovCG(setting, data, regpar=alpha/256)
solver = Landweber(setting, data,grid.zeros())
data2 = data.copy()
#solver = TikhonovAlphaGrid(setting,data,iter([1e-2,3e-3, 1e-3]))
max_its= 2
setting2 = RegularizationSetting(op=conv, penalty=L2, data_fid=L2)
stoprule =  (rules.CountIterations(max_iterations=max_its)
   +rules.Discrepancy(setting2.h_codomain.norm, data2,
        noiselevel=setting.h_codomain.norm(np.sqrt(data2[:])), tau=1.0)
)
reco, reco_data = solver.run(stoprule)
print('relative reconstruction error:', np.linalg.norm(reco[:]-exact_sol[:])/np.linalg.norm(exact_sol[:]))
comparison_plot(grid,exact_sol,reco,title_left="Landweber {} its".format(solver.iteration_step_nr))


In [ ]:
largebox = np.logical_and(abs(X)<=0.8,(abs(Y+0.25)<=1.))
plt.imshow(largebox.T)
emb = CoordinateProjection(grid,largebox).adjoint

In [ ]:
alpha = 2e-7
pen = HmDomain(grid, largebox, index = 1)
setting = RegularizationSetting(op=conv*emb, penalty=pen, data_fid=L2)
solver = TikhonovCG(setting, data,alpha/256,reltolx=0.001, reltoly=0.001,tol=1e-8)
stoprule = rules.CountIterations(max_iterations=400)
reco, reco_data = solver.run(stoprule)
comparison_plot(grid,exact_sol,emb(reco),title_left="Tikhonov Sobolev penalty")